# segment0: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('06_segment0_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,98.747,98.605,99.697,97.576,99.949,98.763,99.812,99.966,98.676
smote,98.723,98.457,99.398,97.576,99.899,98.737,99.876,99.977,98.590
bsmote,98.918,96.671,94.722,98.779,99.065,98.922,99.729,99.951,97.794
ros,98.955,98.694,99.394,98.026,99.899,98.962,99.903,99.983,98.824
adasyn,99.300,97.059,94.751,99.538,99.065,99.302,99.872,99.979,98.179
casb,99.186,97.469,95.961,99.086,99.293,99.189,99.955,99.992,98.327
casb_relaxed,99.210,98.705,98.812,98.634,99.798,99.216,99.975,99.996,98.958
casb_edited,98.993,98.183,98.099,98.329,99.672,99.000,99.964,99.994,98.588
casb_ros,99.340,97.273,95.155,99.541,99.141,99.341,99.933,99.989,98.306


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,98.652,97.281,96.726,97.879,99.444,98.662,99.041,99.641,97.967
smote,98.962,96.531,94.270,98.939,98.989,98.964,98.231,99.578,97.746
bsmote,98.987,96.330,93.762,99.091,98.888,98.990,98.263,99.581,97.658
ros,98.836,96.164,93.700,98.788,98.888,98.838,98.375,99.587,97.500
adasyn,99.063,96.051,92.955,99.394,98.737,99.065,98.010,99.562,97.557
casb,98.836,95.822,92.927,98.939,98.737,98.838,97.934,99.552,97.329
casb_relaxed,98.924,95.621,92.280,99.242,98.610,98.926,98.107,99.569,97.272
casb_edited,98.911,95.553,92.158,99.242,98.585,98.914,97.647,99.526,97.232
casb_ros,98.962,95.834,92.683,99.242,98.686,98.964,98.255,99.581,97.398


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,99.146,99.077,99.846,98.329,99.975,99.152,99.944,99.990,99.112
smote,99.338,99.082,99.387,98.783,99.899,99.341,99.927,99.984,99.210
bsmote,99.620,99.241,99.095,99.394,99.848,99.621,99.963,99.993,99.431
ros,99.351,99.158,99.541,98.783,99.924,99.354,99.959,99.992,99.254
adasyn,99.620,99.241,99.095,99.394,99.848,99.621,99.952,99.991,99.431
casb,99.556,99.239,99.245,99.240,99.874,99.557,99.942,99.988,99.397
casb_relaxed,99.479,99.161,99.245,99.089,99.874,99.481,99.957,99.992,99.320
casb_edited,99.696,99.318,99.095,99.545,99.848,99.697,99.941,99.988,99.507
casb_ros,99.645,99.392,99.394,99.394,99.899,99.646,99.954,99.991,99.519


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,99.186,98.927,99.394,98.485,99.899,99.192,99.128,99.904,99.056
smote,99.186,98.927,99.394,98.485,99.899,99.192,99.419,99.925,99.056
bsmote,99.316,96.768,94.032,99.697,98.939,99.318,99.863,99.978,98.042
ros,99.186,98.927,99.394,98.485,99.899,99.192,99.475,99.940,99.056
adasyn,99.430,97.427,95.300,99.697,99.166,99.432,99.817,99.973,98.429
casb,99.366,98.496,97.940,99.091,99.646,99.369,99.778,99.971,98.931
casb_relaxed,99.173,98.851,99.242,98.485,99.874,99.179,99.709,99.963,99.012
casb_edited,99.148,98.701,98.934,98.485,99.823,99.154,99.691,99.960,98.925
casb_ros,99.467,98.721,98.234,99.242,99.697,99.470,99.780,99.971,99.094


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,98.933,98.472,98.916,98.067,99.817,98.942,99.481,99.875,98.703
smote,99.052,98.249,98.112,98.446,99.672,99.059,99.363,99.866,98.650
bsmote,99.210,97.252,95.403,99.240,99.185,99.213,99.454,99.876,98.231
ros,99.082,98.236,98.007,98.520,99.653,99.087,99.428,99.876,98.659
adasyn,99.353,97.445,95.525,99.506,99.204,99.355,99.413,99.876,98.399
casb,99.236,97.756,96.518,99.089,99.387,99.238,99.402,99.876,98.496
casb_relaxed,99.197,98.085,97.395,98.862,99.539,99.201,99.437,99.880,98.641
casb_edited,99.187,97.939,97.072,98.900,99.482,99.191,99.311,99.867,98.563
casb_ros,99.353,97.805,96.366,99.355,99.356,99.355,99.480,99.883,98.579


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,99.340,adasyn,99.300,0.040,WIN
1,f1,casb_relaxed,98.705,ros,98.694,0.012,WIN
2,precision,casb_relaxed,98.812,smote,99.398,-0.586,LOSS
3,recall,casb_ros,99.541,adasyn,99.538,0.002,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,98.962,adasyn,99.063,-0.101,LOSS
1,f1,casb_ros,95.834,smote,96.531,-0.696,LOSS
2,precision,casb,92.927,smote,94.270,-1.344,LOSS
3,recall,casb_relaxed,99.242,adasyn,99.394,-0.152,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,99.696,bsmote,99.620,0.076,WIN
1,f1,casb_ros,99.392,bsmote,99.241,0.150,WIN
2,precision,casb_ros,99.394,ros,99.541,-0.147,LOSS
3,recall,casb_edited,99.545,bsmote,99.394,0.152,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,99.353,adasyn,99.353,0.000,WIN
1,f1,casb_relaxed,98.085,smote,98.249,-0.164,LOSS
2,precision,casb_relaxed,97.395,smote,98.112,-0.718,LOSS
3,recall,casb_ros,99.355,adasyn,99.506,-0.151,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.992395,0.992366,1.000000,0.984848,1.000000,0.992424,0.998362,0.999694,0.992381,None
1,42,1,svm,smote,0.50,0.992395,0.992366,1.000000,0.984848,1.000000,0.992424,0.998737,0.999770,0.992381,None
2,42,1,svm,bsmote,0.50,0.991122,0.949640,0.904110,1.000000,0.982323,0.991162,0.999554,0.999923,0.970381,None
3,42,1,svm,ros,0.50,0.992395,0.992366,1.000000,0.984848,1.000000,0.992424,0.998737,0.999770,0.992381,None
4,42,1,svm,adasyn,0.50,0.991122,0.949640,0.904110,1.000000,0.982323,0.991162,0.999554,0.999923,0.970381,None
5,42,1,svm,casb,0.50,0.996205,0.977778,0.956522,1.000000,0.992424,0.996212,1.000000,1.000000,0.986991,None
6,42,1,svm,casb_relaxed,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
7,42,1,svm,casb_edited,0.50,0.998737,0.992481,0.985075,1.000000,0.997475,0.998737,1.000000,1.000000,0.995609,None
8,42,1,svm,casb_ros,0.50,0.993667,0.963504,0.929577,1.000000,0.987374,0.993687,1.000000,1.000000,0.978585,None
9,42,1,knn,original,0.50,0.989886,0.977444,0.970149,0.984848,0.994949,0.989899,0.999554,0.999923,0.983665,None
